# Customer Segmentation Project
**Author:** Abhinav Gaur
**Dataset:** Superstore Sales (Kaggle: rohitsahoo/sales-forecasting)
**Goal:** Segment customers by purchase behavior (RFM) and demographics (customer segment and region) using K-Means clustering.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv("train.csv")
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df = df.drop_duplicates()
print(df["Customer ID"].nunique(), "unique customers")

793 unique customers


## Feature engineering
For every customer I calculated three RFM features:
- **Recency:** days since the customer's last order
- **Frequency:** number of orders placed
- **Monetary:** total amount spent

The dataset has no age or income, so the demographic details used are the customer Segment (Consumer, Corporate, Home Office) and Region.

In [2]:
snapshot = df["Order Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("Customer ID").agg(
    Recency=("Order Date", lambda x: (snapshot - x.max()).days),
    Frequency=("Order ID", "nunique"),
    Monetary=("Sales", "sum")
).reset_index()

demo = df.groupby("Customer ID").agg(
    Segment=("Segment", "first"),
    Region=("Region", "first")
).reset_index()

rfm = rfm.merge(demo, on="Customer ID")
rfm.describe()

,Recency,Frequency,Monetary
count,793.000000,793.000000,793.000000
mean,149.286255,6.206810,2851.874884
std,187.081466,2.525647,2620.668723
min,1.000000,1.000000,4.833000
25%,31.000000,4.000000,1081.466000
50%,76.000000,6.000000,2215.002000
75%,185.000000,8.000000,3670.258000
max,1166.000000,17.000000,25043.050000


## Scaling
Frequency and Monetary are heavily skewed, so I applied a log transform, then standardized all three features so that none dominates the clustering.

In [3]:
from sklearn.preprocessing import StandardScaler

X = rfm[["Recency", "Frequency", "Monetary"]].copy()
X["Frequency"] = np.log1p(X["Frequency"])
X["Monetary"] = np.log1p(X["Monetary"])

X_scaled = StandardScaler().fit_transform(X)

## Choosing the number of clusters
The elbow method and silhouette score were used to choose k. The elbow chart bends around k = [your k], and the silhouette score is [high / reasonable] there, so I selected [your k] clusters.


In [7]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=42)
rfm["Cluster"] = km.fit_predict(X_scaled).astype(str)

profile = rfm.groupby("Cluster").agg(
    Customers=("Customer ID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Spend=("Monetary", "mean")
).round(1)
profile

,Customers,Avg_Recency,Avg_Frequency,Avg_Spend
Cluster,,,,
0,286,104.7,4.9,1658.8
1,89,576.7,3.8,1771.1
2,47,175.4,2.4,274.0
3,371,77.8,8.2,4357.5


## Cluster profiles
- **Cluster 0 – [name]:** [number] customers, average recency [x] days, [x] orders, average spend $[x]. [One line: e.g. "Recent, frequent, high-spending customers."]
- **Cluster 1 – [name]:** ...
- **Cluster 2 – [name]:** ...
- **Cluster 3 – [name]:** ...


In [8]:
px.scatter(rfm, x="Frequency", y="Monetary", color="Cluster",
           hover_data=["Customer ID", "Recency"],
           title="Customer Segments: Frequency vs Spend").show()

px.scatter_3d(rfm, x="Recency", y="Frequency", z="Monetary",
              color="Cluster", title="Customer Segments (3D)").show()

## Visualization
The scatter plots show how the clusters separate by frequency and spend. [One line, e.g. "Loyal customers sit at the top right with high orders and high spending, while inactive customers cluster at the bottom left."]

In [9]:
px.histogram(rfm, x="Segment", color="Cluster", barmode="group",
             title="Clusters by Customer Segment").show()
px.histogram(rfm, x="Region", color="Cluster", barmode="group",
             title="Clusters by Region").show()

## Demographics
[Which segment (Consumer, Corporate, Home Office) is most common in each cluster? Which region has the most high-value customers?] Example: "Consumers make up the largest share of every cluster, and the West region has the most loyal customers."

## Conclusion and recommendations
- **Reward loyal high-value customers** with loyalty programs and early access to new products.
- **Win back inactive customers** with discount offers and reminder emails.
- **Encourage new customers** to buy again with follow-up offers.
- **Target big occasional spenders** with bundles and premium recommendations.

Segmenting customers lets the business focus its marketing budget on the groups that matter most.

**Tools used:** Python, Pandas, scikit-learn (K-Means), Plotly.